## 2. Data Cleaning and Preparation

### 2.1 Importing all the necessary Libraries needed.

In [72]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

### 2.2 Loading and inspecting dataset structure

In [73]:
neiss_2024 = pd.read_excel('../raw_data/neiss_2024_raw.xlsx')

print("Shape:", neiss_2024.shape)
neiss_2024.info()

Shape: (361672, 25)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 361672 entries, 0 to 361671
Data columns (total 25 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   CPSC_Case_Number   361672 non-null  int64         
 1   Treatment_Date     361672 non-null  datetime64[ns]
 2   Age                361672 non-null  int64         
 3   Sex                361672 non-null  int64         
 4   Race               361672 non-null  int64         
 5   Other_Race         15277 non-null   object        
 6   Hispanic           361672 non-null  int64         
 7   Body_Part          361672 non-null  int64         
 8   Diagnosis          361672 non-null  int64         
 9   Other_Diagnosis    77535 non-null   object        
 10  Body_Part_2        88551 non-null   float64       
 11  Diagnosis_2        88551 non-null   float64       
 12  Other_Diagnosis_2  22830 non-null   object        
 13  Disposition        36167

### 2.3 Standardizing column names(lowercase)

In [74]:
neiss_2024_clean = neiss_2024.copy()
neiss_2024_clean.columns = neiss_2024.columns.str.lower()
neiss_2024_clean.head(2).T

,0,1
cpsc_case_number,240108461,240108462
treatment_date,2024-01-01 00:00:00,2024-01-01 00:00:00
age,16,56
sex,1,2
race,0,1
other_race,NaN,NaN
hispanic,1,2
body_part,30,92
diagnosis,64,59
other_diagnosis,NaN,NaN


### 2.4 Handling missing and coded values (0 = not recorded)

In [75]:
missing_zero_cols = ['sex', 'race', 'hispanic', 'location']

for col in missing_zero_cols:
    neiss_2024_clean[col] = (neiss_2024_clean[col].replace(0, pd.NA).astype('category'))


### 2.5 Categorical mapping

In [76]:
def categorical_mapping(df, col, mapping):
  df[col] = df[col].map(mapping)
  df[col] = df[col].str.lower().astype('category')
  return df[col].unique()

#### 2.5.1 Mapping race of the patient as recorded in the ED.

In [77]:
race_map = {1: 'white', 2: 'black/african american',
            3: 'other', 4: 'asian', 5: 'american indian/alaska native',
            6: 'native hawaiian/pacific islander'}

categorical_mapping(neiss_2024_clean, 'race', race_map)

[NaN, 'white', 'black/african american', 'asian', 'other', 'american indian/alaska native', 'native hawaiian/pacific islander']
Categories (6, object): ['american indian/alaska native', 'asian', 'black/african american',
                         'native hawaiian/pacific islander', 'other', 'white']

#### 2.5.2 Mapping self-reported sex of the patient

In [78]:
gender_map = {1: 'male', 2: 'female'}

categorical_mapping(neiss_2024_clean, 'sex', gender_map)

['male', 'female', NaN]
Categories (2, object): ['female', 'male']

#### 2.5.3 Mapping whether the patient is Hispanic/Latino

In [79]:
hispanic_map = {1: 'yes', 2: 'no'}

categorical_mapping(neiss_2024_clean, 'hispanic', hispanic_map)

['yes', 'no', NaN]
Categories (2, object): ['no', 'yes']

#### 2.5.4 Mapping primary injured body part

In [80]:
neiss_2024_clean['body_part'] = (neiss_2024_clean['body_part'].replace({87: np.nan}))

body_part_map = { 0 : 'Internal', 30: 'Shoulder', 31 : 'Upper Trunk',
                  32 : 'Elbow', 33 : 'Lower Arm', 34 : 'Wrist',
                  35 : 'Knee', 36 : 'Lower Leg', 37 : 'Ankle',
                  38 : 'Pubic Region', 75 : 'Head',76 : 'Face',
                  77 : 'Eyeball', 79 : 'Lower Trunk', 80 : 'Upper Arm',
                  81 : 'Upper Leg', 82 : 'Hand', 83 : 'Foot',
                  84 : '25-50% of Body', 85 : 'All Parts Body', 88 : 'Mouth',
                  89 : 'Neck', 92 : 'Finger', 93 : 'Toe', 94 : 'Ear'}


categorical_mapping(neiss_2024_clean, 'body_part', body_part_map)


['shoulder', 'finger', 'toe', 'internal', 'face', ..., 'mouth', 'wrist', 'upper leg', 'pubic region', '25-50% of body']
Length: 26
Categories (25, object): ['25-50% of body', 'all parts body', 'ankle', 'ear', ..., 'upper arm', 'upper leg',
                          'upper trunk', 'wrist']

#### 2.5.5 Mapping secondary injured body part (if applicable)

In [81]:
categorical_mapping(neiss_2024_clean, 'body_part_2', body_part_map)

[NaN, 'lower trunk', 'lower leg', 'head', 'eyeball', ..., 'toe', 'elbow', 'ankle', 'upper leg', 'pubic region']
Length: 25
Categories (24, object): ['all parts body', 'ankle', 'ear', 'elbow', ..., 'upper arm', 'upper leg',
                          'upper trunk', 'wrist']

#### 2.5.6 Mapping primary injury diagnosis

In [82]:
diagnosis_map = { 41: 'Ingestion', 42: 'Aspiration', 46: 'Burns, Electrical',
                  47: 'Burns, Not Specified', 48: 'Burns, Scald', 49: 'Burns, Chemical',
                  50: 'Amputaion', 51: 'Burns, Thermal', 52: 'Concussions',
                  53: 'Contusions, Abrasions', 54: 'Crushing', 55: 'Dislocation',
                  56: 'Foreign Body', 57: 'Fracture', 58: 'Hematoma', 59: 'Laceration',
                  60: 'Dental Injury' ,61: 'Nerve Damage' ,62: 'Internal Organ Injury',
                  63: 'Puncture' ,64: 'Strain, Sprain' ,65: 'Anoxia' ,66: 'Hemorrhage',
                  67: 'Electric Shock' ,68: 'Poisoning' ,69: 'Submersion',
                  71: 'Other/Not Stated' ,72: 'Avulsion' ,73: 'Burns, Radiation' ,74: 'Dermatitis, Conjunctivitis'}

categorical_mapping(neiss_2024_clean, 'diagnosis', diagnosis_map)


['strain, sprain', 'laceration', 'crushing', 'ingestion', 'hematoma', ..., 'anoxia', 'burns, electrical', 'submersion', 'burns, not specified', 'burns, radiation']
Length: 30
Categories (30, object): ['amputaion', 'anoxia', 'aspiration', 'avulsion', ..., 'poisoning',
                          'puncture', 'strain, sprain', 'submersion']

#### 2.5.7 Mapping secondary injury diagnosis (if applicable)

In [83]:
categorical_mapping(neiss_2024_clean, 'diagnosis_2', diagnosis_map)

[NaN, 'other/not stated', 'fracture', 'internal organ injury', 'laceration', ..., 'electric shock', 'burns, chemical', 'burns, electrical', 'burns, radiation', 'submersion']
Length: 31
Categories (30, object): ['amputaion', 'anoxia', 'aspiration', 'avulsion', ..., 'poisoning',
                          'puncture', 'strain, sprain', 'submersion']

#### 2.5.8 Mapping patient outcome after ED visit

In [84]:
neiss_2024_clean['disposition'] = (neiss_2024_clean['disposition'].replace({9: np.nan}))

disposition_map = { 1: 'Treated/Examined and Released',
                   2: 'Treated and Transferred',
                   4: 'Treated and Admitted/Hospitalized',
                   5: 'Held for Observation',
                   6: 'Left Without Being Seen',
                   8: 'Fatality, Incl. DOA, Died in ER'}

categorical_mapping(neiss_2024_clean, 'disposition', disposition_map)

['treated/examined and released', 'treated and transferred', 'treated and admitted/hospitalized', 'left without being seen', 'held for observation', 'fatality, incl. doa, died in er', NaN]
Categories (6, object): ['fatality, incl. doa, died in er', 'held for observation', 'left without being seen',
                         'treated and admitted/hospitalized', 'treated and transferred',
                         'treated/examined and released']

#### 2.5.9 Mapping location where the injury occurred

In [85]:
location_map = {1: 'Home', 2: 'Farm/Ranch', 4: 'Street or Highway',
                5: 'Other Public Property', 6: 'Mobile/Manufactured Home',
                7: 'Industrial', 8: 'School/Daycare', 9: 'Place of Recreation or Sports'}

categorical_mapping(neiss_2024_clean, 'location', location_map)

['place of recreation or sports', 'home', NaN, 'other public property', 'street or highway', 'school/daycare', 'farm/ranch', 'mobile/manufactured home', 'industrial']
Categories (8, object): ['farm/ranch', 'home', 'industrial', 'mobile/manufactured home',
                         'other public property', 'place of recreation or sports', 'school/daycare',
                         'street or highway']

#### 2.5.10 Mapping whether fire was involved in the incident

In [86]:
fire_involvement_map = {
    0:'no_fire_or_not_recorded',
    1:'fire_fd_attended',
    2:'fire_fd_not_attended',
    3:'fire_fd_unknown'
}

categorical_mapping(neiss_2024_clean, 'fire_involvement', fire_involvement_map)

['no_fire_or_not_recorded', 'fire_fd_unknown', 'fire_fd_attended', 'fire_fd_not_attended']
Categories (4, object): ['fire_fd_attended', 'fire_fd_not_attended', 'fire_fd_unknown',
                         'no_fire_or_not_recorded']

#### 2.5.11 Mapping alcohol consumption related to the incident

In [87]:
alcohol_map = {0: 'No/No Information', 1: 'Yes'}

categorical_mapping(neiss_2024_clean, 'alcohol', alcohol_map)

['no/no information', 'yes']
Categories (2, object): ['no/no information', 'yes']

#### 2.5.12 Mapping drug or medication involvement in the incident.

In [88]:
drug_map = {0: 'No/No Information', 1: 'Yes'}

categorical_mapping(neiss_2024_clean, 'drug', drug_map)

['no/no information', 'yes']
Categories (2, object): ['no/no information', 'yes']

### 2.6 Transforming to age by year

In [100]:
neiss_2024_clean['age_year'] = (neiss_2024_clean['age'].
                                replace(0, pd.NA).
                                astype('Float64'))

months_age = neiss_2024_clean['age_year'].between(201, 223)

neiss_2024_clean.loc[months_age, "age_year"] = (neiss_2024_clean.loc[months_age, "age_year"] - 200) / 12

neiss_2024_clean['age_year'].value_counts(dropna=False, ascending=False )

,count
age_year,
2.0,12087
3.0,9883
14.0,8565
13.0,8544
4.0,8441
...,...
104.0,22
105.0,10
106.0,7


### 2.7 Post-cleaning data validation

#### 2.7.1 Inspecting the final outcome of the first 5 rows after cleaning

In [109]:
neiss_2024_clean.head().T

,0,1,2,3,4
cpsc_case_number,240108461,240108462,240109863,240109864,240109865
treatment_date,2024-01-01 00:00:00,2024-01-01 00:00:00,2024-01-01 00:00:00,2024-01-01 00:00:00,2024-01-01 00:00:00
age,16,56,10,3,2
sex,male,female,male,female,male
race,NaN,white,white,NaN,NaN
other_race,NaN,NaN,NaN,NaN,NaN
hispanic,yes,no,no,no,yes
body_part,shoulder,finger,toe,internal,face
diagnosis,"strain, sprain",laceration,crushing,ingestion,laceration
other_diagnosis,NaN,NaN,NaN,NaN,NaN


#### 2.7.2 Inspecting for the number of unique values after cleaning

In [102]:
neiss_2024_clean.nunique()

,0
cpsc_case_number,361672
treatment_date,366
age,131
sex,2
race,6
other_race,129
hispanic,2
body_part,25
diagnosis,30
other_diagnosis,4798


#### 2.7.3 Inspecting for missing values after cleaning

In [105]:
neiss_2024_clean.isnull().sum()

,0
cpsc_case_number,0
treatment_date,0
age,0
sex,356
race,58471
other_race,346395
hispanic,51009
body_part,9554
diagnosis,0
other_diagnosis,284137


#### 2.7.4 Inspecting for for detailed information after cleaning

In [106]:
neiss_2024_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 361672 entries, 0 to 361671
Data columns (total 26 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   cpsc_case_number   361672 non-null  int64         
 1   treatment_date     361672 non-null  datetime64[ns]
 2   age                361672 non-null  int64         
 3   sex                361316 non-null  category      
 4   race               303201 non-null  category      
 5   other_race         15277 non-null   object        
 6   hispanic           310663 non-null  category      
 7   body_part          352118 non-null  category      
 8   diagnosis          361672 non-null  category      
 9   other_diagnosis    77535 non-null   object        
 10  body_part_2        87279 non-null   category      
 11  diagnosis_2        88551 non-null   category      
 12  other_diagnosis_2  22830 non-null   object        
 13  disposition        361669 non-null  category

### 2.8 Final analytical dataset
- The resulting analytical dataset was used for all subsequent exploratory, statistical, and predictive analyses.


In [108]:
analysis_cols = [
    'cpsc_case_number',
    'age_year',
    'sex',
    'race',
    'hispanic',
    'body_part',
    'diagnosis',
    'location',
    'fire_involvement',
    'alcohol',
    'drug',
    'disposition'
]

neiss_2024_analysis = neiss_2024_clean[analysis_cols].copy()
neiss_2024_analysis.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 361672 entries, 0 to 361671
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   cpsc_case_number  361672 non-null  int64   
 1   age_year          361643 non-null  Float64 
 2   sex               361316 non-null  category
 3   race              303201 non-null  category
 4   hispanic          310663 non-null  category
 5   body_part         352118 non-null  category
 6   diagnosis         361672 non-null  category
 7   location          227735 non-null  category
 8   fire_involvement  361672 non-null  category
 9   alcohol           361672 non-null  category
 10  drug              361672 non-null  category
 11  disposition       361669 non-null  category
dtypes: Float64(1), category(10), int64(1)
memory usage: 9.3 MB


### 2.9 Exporting final analytical dataset

In [ ]:
neiss_2024_analysis.to_csv('../clean_data/neiss_2024_analysis.csv',index=False)